# DCR — Análise de risco de mercado

**Carteira estilizada de ativos de investimento próprios | Banco de Portugal**

Este notebook contém os cálculos, os resultados e a interpretação das questões do caso. A limpeza dos ficheiros originais e a construção dos retornos estão em `data_processing.py`.
O ficheiro `analise_aux.py` é um ficheiro auxiliar a este notebook.

**Pressupostos:** 
* Carteira denominada em EUR, ativos em USD sem cobertura cambial, pesos constantes (reequilíbrio diário implícito), 252 dias úteis por ano. 
* Período de calibração: janeiro de 2008 a outubro de 2026 (última observação esperada: 30/10/2026).

## 0 - Preparação


Para configurar o ambiente Python e instalar as dependências do projeto através do uv, consultar o ficheiro README.md.

In [25]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import norm, skew, kurtosis, jarque_bera

from data_processing import df_returns

weights = {
    "us": 0.20,
    "aaa": 0.25,
    "all": 0.40,
    "sp500": 0.05,
    "eurostoxx": 0.10,
}
assets = list(weights)
w = pd.Series(weights, dtype=float)
TRADING_DAYS = 252

assert np.isclose(w.sum(), 1.0)
assert not df_returns[assets].isna().any().any()
assert not df_returns["date"].duplicated().any()

print(f"Observações: {len(df_returns):,}")
print(f"Período efetivo: {df_returns['date'].min():%d/%m/%Y} — {df_returns['date'].max():%d/%m/%Y}")

Observações: 4,636
Período efetivo: 03/01/2008 — 30/10/2026


## 0.1 - Estatísticas descritivas e composição da carteira

As variações das yields soberanas são convertidas em retornos aproximados por `−D_mod × Δyield`, com duração modificada de três anos. Esta aproximação mede essencialmente o efeito-preço, sem cupões, *carry*, *roll-down* ou convexidade. Os retornos das posições dos EUA são expressos em EUR para incorporar o risco cambial.

In [26]:
df_portfolio = df_returns[["date"]].copy()
df_portfolio["portfolio_return"] = df_returns[assets].mul(w).sum(axis=1)

mu_daily = df_portfolio["portfolio_return"].mean()  # estatística descritiva; não usada no VaR/ES
sigma_daily = df_portfolio["portfolio_return"].std()
mu_annual = mu_daily * TRADING_DAYS
sigma_annual = sigma_daily * np.sqrt(TRADING_DAYS)

print(f"Retorno médio anualizado da carteira: {mu_annual:.2%}")
print(f"Volatilidade anualizada da carteira: {sigma_annual:.2%}")

Retorno médio anualizado da carteira: 1.46%
Volatilidade anualizada da carteira: 3.45%


In [27]:
df_volatility = pd.DataFrame({
    "Peso na carteira (%)": w * 100,
    "Volatilidade anual (%)": df_returns[assets].std() * np.sqrt(TRADING_DAYS) * 100,
})
display(df_volatility.round(2))

# Comparação do risco dos ativos norte-americanos antes/depois da conversão cambial
comparison_us = pd.DataFrame({
    "Peso (%)": [weights["us"] * 100, weights["sp500"] * 100],
    "Volatilidade USD (%)": [
        df_returns["us_usd"].std() * np.sqrt(TRADING_DAYS) * 100,
        df_returns["sp500_usd"].std() * np.sqrt(TRADING_DAYS) * 100,
    ],
    "Volatilidade EUR (%)": [
        df_returns["us"].std() * np.sqrt(TRADING_DAYS) * 100,
        df_returns["sp500"].std() * np.sqrt(TRADING_DAYS) * 100,
    ],
}, index=["Obrigações EUA", "S&P 500"])
display(comparison_us.round(2))

,Peso na carteira (%),Volatilidade anual (%)
us,20.0,9.18
aaa,25.0,2.03
all,40.0,2.24
sp500,5.0,20.47
eurostoxx,10.0,22.11


,Peso (%),Volatilidade USD (%),Volatilidade EUR (%)
Obrigações EUA,20.0,2.63,9.18
S&P 500,5.0,19.95,20.47


## 1. Questão 1 - Value-at-Risk e Expected Shortfall

> **Enunciado:** Estime o VaR e a ES da carteira a 95% e 99%, para um horizonte de um ano, e identifique os ativos que mais contribuem para estas medidas.

### 1.1 - Pressupostos e fórmulas

Assume-se uma distribuição normal dos retornos com **retorno esperado nulo para efeitos de VaR/ES**. Não extrapolamos a média amostral diária para um ano. Aplicamos a regra da raiz quadrada do tempo, que pressupõe retornos temporalmente independentes e variância constante:

$$
\mathrm{VaR}_{\alpha,1A}=\Phi^{-1}(\alpha)\,\sigma_p\sqrt{252}
$$

$$
\mathrm{ES}_{\alpha,1A}=\sigma_p\sqrt{252}\,\frac{\phi(\Phi^{-1}(\alpha))}{1-\alpha}
$$

Ambas as métricas são expressas como percentagem positiva do valor inicial da carteira. A projeção anual por `√252` é uma aproximação baseada em retornos diários; não corresponde a uma simulação exata de retornos simples compostos.

In [28]:
confidence_levels = [0.95, 0.99]

var_annual = {
    alpha: norm.ppf(alpha) * sigma_annual
    for alpha in confidence_levels
}
es_annual = {
    alpha: sigma_annual * norm.pdf(norm.ppf(alpha)) / (1 - alpha)
    for alpha in confidence_levels
}

risk_summary = pd.DataFrame({
    "VaR anual (%)": {f"{alpha:.0%}": 100 * var_annual[alpha] for alpha in confidence_levels},
    "ES anual (%)": {f"{alpha:.0%}": 100 * es_annual[alpha] for alpha in confidence_levels},
})
display(risk_summary.round(2))

# Valores diários, opcionais para diagnóstico
var_daily = {alpha: norm.ppf(alpha) * sigma_daily for alpha in confidence_levels}

,VaR anual (%),ES anual (%)
95%,5.68,7.13
99%,8.04,9.21


### 1.2 - Decomposição do risco por ativo

Se `w` é o vetor de pesos e `Σ` a matriz de covariâncias diárias:

$$
\sigma_p = \sqrt{w^\top\Sigma w}, \qquad
\mathrm{MCR}_i = \frac{(\Sigma w)_i}{\sigma_p}
$$

$$
\mathrm{ComponentVaR}_{i,\alpha}=w_i\,\Phi^{-1}(\alpha)\,\sqrt{252}\,\mathrm{MCR}_i
$$

$$
\mathrm{ComponentES}_{i,\alpha}=w_i\,\frac{\phi(\Phi^{-1}(\alpha))}{1-\alpha}\,\sqrt{252}\,\mathrm{MCR}_i
$$

As componentes somam o VaR/ES da carteira. As contribuições relativas são iguais entre VaR e ES porque, neste modelo normal de média nula, ambos são múltiplos positivos da mesma volatilidade.

Onde:
- $w_i$: peso do ativo $i$ na carteira.
- $\Sigma$: matriz de covariâncias dos retornos diários.
- $\sigma_p$: volatilidade diária da carteira.
- $\mathrm{MCR}_i$: contribuição marginal do ativo $i$ para a volatilidade.
- $\alpha$: nível de confiança (95% ou 99%).
- $\Phi^{-1}(\alpha)$: função inversa da distribuição acumulada da normal padrão, que devolve o quantil correspondente ao nível de confiança. Por exemplo, $\Phi^{-1}(0.99) \approx 2.326$.
- $\phi(z)$: função densidade de probabilidade da normal padrão, avaliada no ponto $z$.


In [29]:
cov_matrix = df_returns[assets].cov()
sigma_from_cov = np.sqrt(w @ cov_matrix @ w)
assert np.isclose(sigma_from_cov, sigma_daily)

sigma_p = np.sqrt(w @ cov_matrix @ w)

mcr = (cov_matrix @ w) / sigma_from_cov
component_var = {
    alpha: w * norm.ppf(alpha) * mcr * np.sqrt(TRADING_DAYS)
    for alpha in confidence_levels
}
component_es = {
    alpha: w * norm.pdf(norm.ppf(alpha)) / (1 - alpha) * mcr * np.sqrt(TRADING_DAYS)
    for alpha in confidence_levels
}

for alpha in confidence_levels:
    assert np.isclose(component_var[alpha].sum(), var_annual[alpha])
    assert np.isclose(component_es[alpha].sum(), es_annual[alpha])

contribution_pct = component_var[0.99] / var_annual[0.99] * 100

# Componentes expressas em pontos percentuais do valor inicial da carteira
df_component_var = pd.DataFrame({
    "Peso (%)": w * 100,
    "Component VaR 95% (p.p.)": component_var[0.95] * 100,
    "Component VaR 99% (p.p.)": component_var[0.99] * 100,
    "Contribuição VaR total (%)": contribution_pct,
    "Volatilidade anual (%)": df_returns[assets].std() * np.sqrt(TRADING_DAYS) * 100,
})

df_component_es = pd.DataFrame({
    "Peso (%)": w * 100,
    "Component ES 95% (p.p.)": component_es[0.95] * 100,
    "Component ES 99% (p.p.)": component_es[0.99] * 100,
    "Contribuição ES total (%)": component_es[0.99] / es_annual[0.99] * 100,
    "Volatilidade anual (%)": df_returns[assets].std() * np.sqrt(TRADING_DAYS) * 100,
})

# Linha total: a volatilidade da carteira NÃO é a soma das volatilidades dos ativos
df_component_var.loc["Total (carteira)"] = [100, var_annual[0.95] * 100, var_annual[0.99] * 100, 100, sigma_annual * 100]
df_component_es.loc["Total (carteira)"] = [100, es_annual[0.95] * 100, es_annual[0.99] * 100, 100, sigma_annual * 100]

print("Component VaR")
display(df_component_var.round(2))
print("Component ES")
display(df_component_es.round(2))

Component VaR


,Peso (%),Component VaR 95% (p.p.),Component VaR 99% (p.p.),Contribuição VaR total (%),Volatilidade anual (%)
us,20.0,1.39,1.97,24.54,9.18
aaa,25.0,0.27,0.38,4.67,2.03
all,40.0,0.63,0.89,11.08,2.24
sp500,5.0,1.19,1.69,20.99,20.47
eurostoxx,10.0,2.20,3.11,38.71,22.11
Total (carteira),100.0,5.68,8.04,100.00,3.45


Component ES


,Peso (%),Component ES 95% (p.p.),Component ES 99% (p.p.),Contribuição ES total (%),Volatilidade anual (%)
us,20.0,1.75,2.26,24.54,9.18
aaa,25.0,0.33,0.43,4.67,2.03
all,40.0,0.79,1.02,11.08,2.24
sp500,5.0,1.50,1.93,20.99,20.47
eurostoxx,10.0,2.76,3.56,38.71,22.11
Total (carteira),100.0,7.13,9.21,100.00,3.45


### 1.3. - Interpretação dos resultados

| Medida | 95% | 99% |
|---|---:|---:|
| VaR anual | **5,68%** | **8,04%** |
| ES anual | **7,13%** | **9,21%** |

Segundo o modelo, há uma probabilidade estimada de 5% (1%) de a carteira sofrer uma perda superior a 5,68% (8,04%) num horizonte de um ano. Nos piores 5% (1%) dos cenários, a perda média estimada é de 7,13% (9,21%).

**Principais contribuintes:** Euro STOXX 50 (~38,71%), obrigações soberanas EUA (~24,54%), S&P 500 (~20,99%), obrigações da área do euro - componente de crédito (~11,08%) e obrigações AAA (~4,67%). Os dois índices acionistas representam **15% do capital**, mas cerca de **59,7% do VaR e ES**.

O Euro STOXX 50 apresenta a maior contribuição porque combina volatilidade elevada (~22,11%) com um peso de 10%, superior ao peso de 5% do S&P 500. As correlações com as restantes posições também influenciam a contribuição marginal.

Nas obrigações dos EUA, o facto de não haver cobertura cambial é particularmente relevante: a volatilidade anualizada passa de cerca de **2,63% em USD** para **9,18% em EUR**. Adicionalmente, o peso deste ativo no portfólio (20%) é quatro vezes o do S&P 500 (5%), explicando por que contribui mais para o VaR apesar de ter um perfil de risco mais conservador (vs S&P500)

**Nota:** as contribuições em `Component VaR/ES (p.p.)` representam pontos percentuais da medida de risco da carteira; a coluna `Contribuição (%)` representa a fração do risco total explicada pelo ativo.

## 2. Questão 2 - Diversificação e cenários extremos

> **Enunciado:** Avalie os benefícios de diversificação decorrentes da combinação de obrigações soberanas e ações. Em que medida permanecem válidos em cenários extremos?

In [30]:
corr_matrix = df_returns[assets].corr()
display(corr_matrix.round(3))

,us,aaa,all,sp500,eurostoxx
us,1.000,0.254,0.085,0.190,-0.290
aaa,0.254,1.000,0.725,-0.050,-0.215
all,0.085,0.725,1.000,0.015,0.020
sp500,0.190,-0.050,0.015,1.000,0.492
eurostoxx,-0.290,-0.215,0.020,0.492,1.000


### 2.1 - Benefícios da diversificação
Para avaliar os benefícios da diversificação vamos assumir um pressuposto forte: e se os ativos tivessem todos uma correlação de 1?

Nesse caso não haveria ganhos de diversificação, uma vez que os retornos dos ativos estariam perfeitamente correlacionados e mover-se-iam sempre na mesma direção.

Na célula abaixo calculamos a volatilidade do portfólio sem benefícios de diversificação (correlação =1) e estimamos que os benefícios da diversificação são aproximadamente uma redução de 47% na volatilidade do portfólio.

In [31]:
# Volatilidades anuais individuais
asset_vol = df_returns[assets].std() * np.sqrt(252)

# Volatilidade hipotética com correlações iguais a +1
vol_no_diversification = (w * asset_vol).sum()

# Volatilidade efetiva da carteira
vol_portfolio = sigma_p * np.sqrt(252)

# Benefício da diversificação
diversification_benefit = (
    1 - vol_portfolio / vol_no_diversification
) * 100

print(f"Volatilidade sem diversificação: {vol_no_diversification:.2%}")
print(f"Volatilidade efetiva: {vol_portfolio:.2%}")
print(f"Redução relativa da volatilidade: {diversification_benefit:.2f}%")

Volatilidade sem diversificação: 6.47%
Volatilidade efetiva: 3.45%
Redução relativa da volatilidade: 46.63%


### 2.2 - Estabilidade das correlações entre ações e obrigações

O objetivo desta análise é apurar se as correlações entre ações e obrigações soberanas permanecem estáveis ao longo do tempo, particularmente durante períodos de stress.

Sabemos pela teoria económica que em períodos de aversão ao risco (**flight to safety**), as obrigações consideradas seguras beneficiam de uma maior procura, enquanto que os investidores reduzem a sua exposição a ativos de alto risco, como as ações. Isto leva a correlações negativas entre as duas classes de ativos.

Contudo, esta relação não é constante. Em períodos caracterizados por choques inflacionistas e alterações das expectativas de política monetária, ações e obrigações podem apresentar correlações positivas, reduzindo os benefícios da diversificação. Os desenvolvimentos de 2026, associados às tensões geopolíticas no Médio Oriente, às pressões inflacionistas e à evolução dos mercados acionistas num contexto de crescente investimento em inteligência artificial, reforçam a relevância desta questão.

Para investigar esta dinâmica, serão calculadas correlações móveis de 252 observações entre o Euro STOXX 50 e os três componentes obrigacionistas da carteira, complementadas por uma análise dos períodos de stress de 2008–2009, 2020 e 2022.


### Correlações móveis entre ações e obrigações

Para analisar a estabilidade das correlações ao longo do tempo, calculam-se correlações móveis (*rolling correlations*) de 252 observações diárias entre o Euro STOXX 50 e os três componentes obrigacionistas.

Esta análise permite identificar alterações na relação entre as duas classes de ativos, nomeadamente períodos em que as correlações passam de negativas para positivas, reduzindo os benefícios da diversificação.


In [32]:
window = 252

bond_assets = {
    "us": "Obrigações EUA",
    "aaa": "Obrigações AAA",
    "all": "Obrigações Euro (All)"
}

df_rolling_corr = df_returns[["date"]].copy()

for asset, label in bond_assets.items():
    df_rolling_corr[label] = (
        df_returns["eurostoxx"]
        .rolling(window)
        .corr(df_returns[asset])
    )

fig = px.line(
    df_rolling_corr,
    x="date",
    y=list(bond_assets.values()),
    title="Correlação móvel (252 dias) — Euro STOXX 50 vs. Obrigações",
    labels={
        "date": "Data",
        "value": "Correlação",
        "variable": "Ativo obrigacionista"
    }
)

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_yaxes(range=[-1, 1])
fig.update_layout(hovermode="x unified")

fig.show()

Como podemos ver as correlações começam negativas em janeiro 2009 (como fazemos uma média móvel de 252 diárias, apenas começamos em 2009), contudo em 2012-2013, devido à crise soberana na zona euro, a correlação fica positiva, uma vez que mesmo os ativos seguros (obrigações) eram vistos como arriscados pelos investidores.

Outro detalhe importante que podemos retirar desta análise é o facto das obrigações AAA (de melhor qualidade de rating vs All e sem o risco cambial embutido vs EUA) têm consistentemente uma correlação mais negativa com o EuroStoxx. Isto diz-nos que alguma da correlação positiva entre os outros 2 ativos e o Eurostoxx se deve ao perfil de risco mais elevado destes, que é mais semelhante ao eurostoxx.

### 2.3 - Comparação dos períodos de stress


### Análise de períodos de stress

Para avaliar se os benefícios da diversificação persistem em cenários extremos, analisam-se três episódios históricos:

- **2008–2009:** crise financeira global.
- **2020:** choque associado à pandemia COVID-19.
- **2022:** choque inflacionista e subida das taxas de juro.

Para cada período, calculam-se as correlações entre ações e obrigações, a volatilidade anualizada da carteira e a redução relativa de volatilidade face ao cenário hipotético de correlação perfeita positiva.

Os resultados serão comparados com os obtidos para a amostra completa.


In [33]:

# Definir períodos de análise
periods = {
    "Amostra completa": ("2008-01-01", "2026-10-31"),
    "Crise financeira 2008–2009": ("2008-01-01", "2009-12-31"),
    "COVID-19 (2020)": ("2020-01-01", "2020-12-31"),
    "Choque inflacionista (2022)": ("2022-01-01", "2022-12-31"),
    "2026 (até outubro)": ("2026-01-01", "2026-10-31")
}

results = []

for period_name, (start, end) in periods.items():

    df_period = df_returns.loc[
        df_returns["date"].between(start, end)
    ]

    # Correlações ações/obrigações
    corr = df_period[assets].corr()

    # Retornos e volatilidade da carteira
    portfolio_returns = df_period[assets].mul(w).sum(axis=1)
    vol_portfolio = portfolio_returns.std() * np.sqrt(252)

    # Volatilidade hipotética com correlação perfeita
    asset_vol = df_period[assets].std() * np.sqrt(252)
    vol_no_div = (w * asset_vol).sum()

    # Redução relativa da volatilidade
    benefit = (1 - vol_portfolio / vol_no_div) * 100

    results.append({
        "Período": period_name,
        "Observações": len(df_period),
        "Corr. STOXX / EUA": corr.loc["eurostoxx", "us"],
        "Corr. STOXX / AAA": corr.loc["eurostoxx", "aaa"],
        "Corr. STOXX / All": corr.loc["eurostoxx", "all"],
        "Vol. carteira (%)": vol_portfolio * 100,
        "Vol. sem div. (%)": vol_no_div * 100,
        "Benefício div. (%)": benefit
    })

df_stress = pd.DataFrame(results).set_index("Período")

display(df_stress.round(2))


,Observações,Corr. STOXX / EUA,Corr. STOXX / AAA,Corr. STOXX / All,Vol. carteira (%),Vol. sem div. (%),Benefício div. (%)
Período,,,,,,,
Amostra completa,4636,-0.29,-0.21,0.02,3.45,6.47,46.63
Crise financeira 2008–2009,491,-0.47,-0.42,-0.38,4.70,10.07,53.36
COVID-19 (2020),246,-0.21,-0.33,0.05,4.62,7.39,37.51
Choque inflacionista (2022),248,-0.47,0.05,0.14,4.46,7.99,44.21
2026 (até outubro),206,-0.32,0.54,0.62,3.06,4.87,37.21


Como podemos observar, as correlações entre o Eurostoxx e os EUA mantêm-se consistentemente negativas em todos os períodos analisados na amostra, mesmo em períodos de stress mais focados.

É também interessante que a correlação entre o Eurostoxx e as obrigações AAA se manteve predominantemente negativa (embora mais negativa na crise de 2008), mas em 2022 e 2026 passou a apresentar valores positivos. Este acontecimento reduz os benefícios de diversificação entre estes 2 ativos numa carteira e é um risco recentemente abordado no Financial Stability Review de Novembro de 2026 apresentado pelo BCE "Sovereign bonds have traditionally acted as a hedge in balanced portfolios. With concerns rising about a pick-up in inflation and interest rates, however, stock and bond returns are again positively correlated, with the correlation in the euro area climbing to its highest level since 2005."

Observamos também que a correlação entre o Euro STOXX 50 e as obrigações soberanas europeias, sem distinção por rating, se manteve predominantemente positiva, à exceção da crise de 2008, quando se verificou um movimento acentuado de *flight to safety*.

Esta diferença face às obrigações AAA poderá estar relacionada com a maior exposição ao risco de crédito, mas também com a sensibilidade às taxas de juro e a outros fatores macroeconómicos. Assim, verificamos que nem todos os segmentos obrigacionistas oferecem os mesmos benefícios de diversificação.

Numa nota final podemos observar que em períodos de stress (naturalmente) a volatilidade da carteira é mais alta, na ordem dos 4.5%. Contudo, num cenário hipotético sem benefícios de diversificação, a volatilidade teria sido muito maior, chegando até a níveis de 10% em 2008. Isto demonstra que a diversificação continua a proporcionar benefícios mesmo em períodos de stress. No entanto, esses benefícios não impedem que o risco absoluto da carteira aumente significativamente durante uma crise. 

## 3. Questão 3 — Limitações metodológicas

> **Enunciado:** Identifique as principais limitações da abordagem seguida e explique como procuraria ultrapassá-las num contexto real de quantificação do risco da carteira de ativos de investimento próprios do Banco.

### 3.1 - Limitações da metodologia

A metodologia utilizada apresenta várias limitações, associadas tanto aos pressupostos estatísticos do VaR paramétrico como às simplificações adotadas para representar os ativos da carteira.

**1. Model risk — Distribuição dos retornos**

O modelo assume que os retornos seguem uma distribuição normal. Contudo, a literatura académica demonstra que os retornos dos ativos apresentam frequentemente assimetria (*skewness*) e excesso de kurtosis (*fat tails*), isto é, uma maior frequência de acontecimentos extremos do que seria esperado numa distribuição normal.

Consequentemente, o VaR paramétrico pode subestimar as perdas em cenários extremos.

Outra limitação prende-se com o fenómeno de *volatility clustering*, segundo o qual períodos de elevada volatilidade tendem a ser seguidos por outros períodos de elevada volatilidade. Este comportamento contraria a hipótese de volatilidade constante que utilizámos no modelo.

**2. Parameter risk — Estimação dos parâmetros**

Para calcular o VaR e ES, utilizámos a volatilidade e as correlações estimadas a partir de uma amostra histórica de aproximadamente 19 anos.

Esta abordagem apresenta algumas limitações. Por um lado, observações muito antigas poderão já não representar as condições atuais de mercado. Por outro, a carteira poderá estar exposta a um regime económico específico, como um período de inflação elevada, cujos riscos não são adequadamente captados pela média histórica.

Adicionalmente, como demonstrado na pergunta 2, as correlações entre ações e obrigações não são constantes, podendo apresentar alterações significativas em diferentes contextos económicos.

**3. Event risk — Eventos extremos**

O VaR é estimado a partir da distribuição assumida para os retornos, não garantindo que acontecimentos extremos estejam adequadamente representados.

Choques geopolíticos, crises financeiras ou alterações inesperadas da política monetária podem originar perdas significativamente superiores às estimadas pelo modelo.

**4. Simplificações na avaliação da carteira**

No caso das obrigações, aproximámos os retornos através da modified duration e das variações das taxas spot a três anos.

Esta abordagem não considera efeitos como a convexidade, os cupões e os movimentos não paralelos das curvas de juros, podendo produzir estimativas imprecisas das variações de preço.

Adicionalmente, assumimos pesos constantes, reequilíbrio diário e ausência de custos de transação. Num contexto real, seria também necessário considerar restrições de liquidez, custos de negociação e outros fatores específicos dos instrumentos.

**5. Horizonte temporal**

A anualização do VaR e ES através da regra da raiz quadrada do tempo pressupõe independência temporal dos retornos e variância constante (i.i.d).

Estes pressupostos podem não se verificar, particularmente num horizonte longo de um ano e em períodos caracterizados por alterações significativas da volatilidade.

### 3.2 - Como ultrapassar estas limitações?

Num contexto real de gestão de risco do Banco de Portugal, devemos complementar a metodologia utilizada com outras medidas e técnicas de análise de risco.

**Stress testing:** considero esta uma das ferramentas mais relevantes e mais utilizadas no mundo financeiro hoje em dia (e.g. ICAAP, ILAAP, stress testes anuais ECB/EBA). Através da definição de cenários históricos ou hipotéticos (e.g. COVID-19 ou crise geopolítica), seria possível avaliar o comportamento da carteira perante choques de taxas de juro, alterações dos spreads na dívida soberana (e.g. dívida francesa ser reavaliada com rating junk), quedas acentuadas dos mercados de ações ou movimentos cambiais significativos. Esta abordagem permitiria analisar cenários que não estão necessariamente representados na distribuição histórica dos retornos.

**Modified VaR:** a utilização de uma aproximação de Cornish-Fisher permitiria incorporar a assimetria (skewness) e o excesso de kurtosis dos retornos, reduzindo a dependência da hipótese de normalidade. Contudo, esta metodologia continuaria sujeita a limitações na representação das caudas da distribuição (black swan).

**Modelos de volatilidade:** modelos como Exponentially Weighted Moving Average (EWMA) ou Generalized Autoregressive Conditional Heteroskedasticity (GARCH) permitiriam atribuir maior importância à evolução recente dos mercados e, no caso do GARCH, modelizar explicitamente o *volatility clustering*.

**Simulações de Monte Carlo:** outra alternativa seria simular milhares de trajetórias futuras dos fatores de risco da carteira ao longo de 252 dias úteis, permitindo calcular a distribuição das perdas no horizonte de um ano, sem depender diretamente da regra da raiz quadrada do tempo.

Estas simulações poderiam ser combinadas com modelos de volatilidade condicional, como o GARCH, e distribuições com caudas mais pesadas, permitindo representar melhor a evolução do risco em diferentes condições de mercado. Contudo, os resultados continuariam dependentes dos pressupostos dos modelos utilizados.

**Maximum drawdown:** esta medida permitiria identificar a maior perda acumulada da carteira relativamente a um máximo histórico, bem como a duração dos períodos de recuperação. Apesar de não prever perdas futuras nem representar necessariamente o risco atual, constitui uma medida simples e intuitiva do comportamento histórico da carteira.

**Backtesting:** permite avaliar a qualidade das estimativas do modelo, comparando as perdas efetivamente observadas com o VaR estimado e identificando situações em que as perdas ultrapassam o limite previsto.

**Análise de sensibilidade:** permite avaliar a robustez dos resultados, analisando o impacto de alterações nos pressupostos do modelo, como a janela histórica utilizada, a volatilidade ou a modified duration dos ativos obrigacionistas.

Por fim, seria importante utilizar informação detalhada sobre os instrumentos da carteira, permitindo uma avaliação mais rigorosa dos riscos de taxa de juro, crédito, câmbio e liquidez.

### Conclusão

O VaR e o ES são ferramentas úteis para quantificar o risco de mercado, mas não são suficientes, isoladamente, para compreender todas as exposições de uma carteira.

A gestão de risco implica necessariamente trabalhar com pressupostos e simplificações. O objetivo não deve ser eliminar toda a incerteza, mas compreender as limitações dos modelos e complementá-los com metodologias alternativas, particularmente análises de cenários extremos.

A utilização conjunta de diferentes medidas de risco permite quantificar os riscos que o portfólio acarreta e obter uma visão mais completa das perdas potenciais e apoiar decisões de gestão de risco mais robustas.

Neste último passo calculamos a skewness e excesso de kurtosis dos retornos diários da carteira e aplicamos o teste de Jarque-Bera para avaliar a hipótese de normalidade. O objetivo é demonstrar que o pressuposto de retornos normais que aplicamos ao portfólio é uma simplificação que deve ser tida em conta.

In [34]:
# Retornos diários da carteira
returns = df_portfolio["portfolio_return"].dropna()

# Estatísticas da distribuição
skewness = skew(returns)
excess_kurtosis = kurtosis(returns)  # Fisher: normal = 0

# Teste de normalidade Jarque-Bera
jb_stat, p_value = jarque_bera(returns)

print(f"Skewness: {skewness:.4f}")
print(f"Excesso de kurtosis: {excess_kurtosis:.4f}")
print(f"Jarque-Bera: {jb_stat:.2f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("Rejeita-se a hipótese nula de normalidade (5%).")
else:
    print("Não se rejeita a hipótese nula de normalidade (5%).")


Skewness: -0.2220
Excesso de kurtosis: 3.5370
Jarque-Bera: 2454.66
P-value: 0.000000
Rejeita-se a hipótese nula de normalidade (5%).


Os resultados mostram uma ligeira assimetria negativa (*skewness = -0,22*) e um excesso de kurtosis elevado (3,54), indicando a presença de caudas mais pesadas do que as da distribuição normal. O teste de Jarque-Bera rejeita a hipótese de normalidade (p-value < 0,001), confirmando que este pressuposto é uma simplificação do modelo.